# Gold: fact_orders

## Import Helper Functions

In [1]:
from src.config_loader import load_config
from src.spark_sql_magic import sql
from src.gold.helper import get_changed_customer_ids, get_changed_order_ids
from src.gold.facts.orders import build_fact_orders, build_fact_orders_incremental, validate_fact_orders
from src.watermark import get_last_commit_ts, get_effective_watermark
from src.writers import overwrite_table, replace_by_key

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Load Configs

In [2]:
cfg = load_config()

JOB_NAMES = cfg["spark_jobs"]["jobs"]
CATALOG = cfg["general"]["catalog"]
SILVER_NAMESPACE = cfg["general"]["namespaces"]["silver"]
GOLD_NAMESPACE = cfg["general"]["namespaces"]["gold"]

cfg_orders= cfg["gold"]["fact_orders"]
TARGET_TABLE = cfg_orders["target_table"]
SOURCE_TABLE = cfg_orders["source_table"]
CUSTOMERS_TABLE = cfg_orders["customers_table"]
DATE_TABLE = cfg_orders["date_table"]
BUFFER_HOURS = cfg_orders["buffer_hours"]
KEY_COLUMNS = cfg_orders["key_columns"]

## Import Libraries and Start Session

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["gold_orders"])
        .getOrCreate()
)

## Run Pipeline

In [4]:
def run_fact_orders_pipeline(spark):
    print("[START] fact_orders pipeline")

    last_commit_ts = get_last_commit_ts(spark, TARGET_TABLE)
    print(f"[INFO] last_commit_ts = {last_commit_ts}")

    if last_commit_ts is None:
        print("[INFO] first run → full rebuild")
        df = build_fact_orders(spark, SOURCE_TABLE, CUSTOMERS_TABLE, DATE_TABLE)
        validate_fact_orders(df)
        overwrite_table(df, TARGET_TABLE)
        print("[END] full rebuild complete")
        return
    
    effective_ts = get_effective_watermark(last_commit_ts, BUFFER_HOURS)
    changed_customer_ids = get_changed_customer_ids(spark, effective_ts)
    changed_order_ids = get_changed_order_ids(spark, effective_ts)

    if changed_customer_ids.isEmpty() and changed_order_ids.isEmpty():
        print("[INFO] no changes detected → skip")
        return

    print("[INFO] changes detected → incremental run")
    df = build_fact_orders_incremental(
        spark,
        SOURCE_TABLE,
        CUSTOMERS_TABLE,
        DATE_TABLE,
        changed_customer_ids,
        changed_order_ids,
    )
    validate_fact_orders(df)
    replace_by_key(spark, df, TARGET_TABLE, KEY_COLUMNS)
    print("[END] incremental update complete")

In [5]:
# if __name__ == "__main__":
#     from pyspark.sql import SparkSession

#     spark = SparkSession.builder.getOrCreate()
run_fact_orders_pipeline(spark)

[START] fact_orders pipeline
[INFO] last_commit_ts = None
[INFO] first run → full rebuild


[END] full rebuild complete


## Sanity Check

In [6]:
%%sql
SHOW TABLES IN polaris.gold;

+---------+------------------+-----------+
|namespace|tableName         |isTemporary|
+---------+------------------+-----------+
|gold     |fact_orders       |false      |
|gold     |dim_date          |false      |
|gold     |dim_customers_scd2|false      |
|gold     |dim_sellers_scd2  |false      |
|gold     |dim_products_scd2 |false      |
+---------+------------------+-----------+



In [7]:
%%sql
SELECT * FROM polaris.gold.fact_orders
LIMIT 10

+--------------------------------+----------------------------------------------------------------+--------------------------------+------------+----------------------+------------------------+-------------------+----------------------------+--------------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_sk                                                     |customer_id                     |order_status|order_purchase_date_sk|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date_sk|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+----------------------------------------------------------------+--------------------------------+------------+----------------------+------------------------+-------------------+----------------------------+--------------------------------+-----------------------------+-----------------------

In [8]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop() 